# Period-finding cost–accuracy frontier

This notebook answers a narrow operational question: **what is the cheapest period strategy that preserves enough accuracy for MALCA?**

It compares the complete spectrum from zero-compute catalog routing through stored production estimates, individual period searches, production consensus, deterministic multi-method arbitration, and the learned V3 selector. It keeps three scientific quantities separate:

1. **Coverage** — the fraction of sources receiving a period.
2. **Selected-period agreement** — exact and harmonic-family agreement for the one period actually selected.
3. **Candidate oracle coverage** — whether an ensemble generated a correct candidate before arbitration.

Catalog-only strategies are never scored against the same catalog consensus used as their input. Real-source agreement and injection-recovery truth are reported separately.

In [ ]:
from __future__ import annotations

from dataclasses import replace
import json
import os
from pathlib import Path
import time

for _name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS", "NUMBA_NUM_THREADS"):
    os.environ.setdefault(_name, "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed, parallel_config

from malca.core.event_epochs import parse_run_epochs_json
from malca.core.phase import align_v_to_g_magnitude
from malca.core.utils import read_lc_dat2
from malca.evaluation.period_cost_accuracy import (
    DEFAULT_HARMONIC_FACTORS,
    add_catalog_reference,
    evaluate_stored_strategies,
    load_review_period_snapshot,
    mark_pareto_frontier,
    period_match_arrays,
    project_runtime_days,
    stratified_runtime_sample,
)
from malca.evaluation.period_candidate_methods import (
    PeriodCandidateMethodsConfig,
    build_candidate_bank,
    refine_scored_candidates,
    run_global_period_searches,
    score_candidate_bank,
    select_scoring_shortlist,
)
from malca.evaluation.period_candidate_ranker import (
    label_candidates,
    load_candidate_ranker_artifact,
    rank_deterministic_baseline,
    score_candidate_ranker,
    select_trial_solutions,
)
from malca.stv.periodicity_gate import prepare_periodicity_lightcurve
from malca.review.period_search import run_pipeline_period_search_for_payload

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "malca").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the MALCA checkout")


ROOT = find_repo_root(Path.cwd().resolve())
RUN_ROOT = ROOT / "output/runs/dat3-full-extended_2026-07-01-v4"
REVIEW_DB = RUN_ROOT / "review/review.db"
V3_ROOT = ROOT / "output/evaluation/period_arbitration/multimethod_candidate_bank/multimethod_discovery_v3_20260725a"
BENCHMARK_TAG = os.environ.get("MALCA_PERIOD_FRONTIER_TAG", "july1_20260730_v1")
if Path(BENCHMARK_TAG).name != BENCHMARK_TAG:
    raise ValueError("MALCA_PERIOD_FRONTIER_TAG must be one path-safe name")
OUTPUT_ROOT = ROOT / "output/evaluation/period_cost_accuracy_frontier" / BENCHMARK_TAG
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Run a fresh representative benchmark by default. Set any RUN_* environment
# variable to 0 when only the stored audit is wanted.
FRESH_SAMPLE_N = int(os.environ.get("MALCA_PERIOD_FRONTIER_SAMPLE_N", "128"))
V3_SAMPLE_N = int(os.environ.get("MALCA_PERIOD_FRONTIER_V3_SAMPLE_N", "16"))
WORKERS = int(os.environ.get("MALCA_PERIOD_FRONTIER_WORKERS", "12"))
INNER_THREADS_PER_WORKER = int(os.environ.get("MALCA_PERIOD_FRONTIER_INNER_THREADS", "1"))
BATCH_SIZE = int(os.environ.get("MALCA_PERIOD_FRONTIER_BATCH_SIZE", "64"))
SHARD_INDEX = int(os.environ.get("MALCA_PERIOD_FRONTIER_SHARD_INDEX", "0"))
SHARD_COUNT = int(os.environ.get("MALCA_PERIOD_FRONTIER_SHARD_COUNT", "1"))
WRITE_AGGREGATES = os.environ.get(
    "MALCA_PERIOD_FRONTIER_WRITE_AGGREGATES",
    "1" if SHARD_COUNT == 1 else "0",
) == "1"
RUN_COMPONENT_BENCHMARK = os.environ.get("MALCA_PERIOD_FRONTIER_RUN_COMPONENTS", "1") == "1"
RUN_PRODUCTION_BENCHMARK = os.environ.get("MALCA_PERIOD_FRONTIER_RUN_PRODUCTION", "1") == "1"
RUN_FULL_V3_BENCHMARK = os.environ.get("MALCA_PERIOD_FRONTIER_RUN_V3", "1") == "1"

SURVEY_SOURCE_COUNT = 17_000_000
PROJECTION_WORKERS_PER_MACHINE = 60
PROJECTION_MACHINES = 6
PROJECTION_EFFICIENCIES = (1.0, 0.8, 0.6)
MATCH_TOLERANCE = 0.05

assert 0 <= SHARD_INDEX < SHARD_COUNT
assert WORKERS >= 1
assert INNER_THREADS_PER_WORKER >= 1
print({
    "root": str(ROOT),
    "benchmark_tag": BENCHMARK_TAG,
    "workers": WORKERS,
    "inner_threads_per_worker": INNER_THREADS_PER_WORKER,
    "sample_n": FRESH_SAMPLE_N,
    "shard": f"{SHARD_INDEX}/{SHARD_COUNT}",
    "write_aggregates": WRITE_AGGREGATES,
    "run_components": RUN_COMPONENT_BENCHMARK,
    "run_production": RUN_PRODUCTION_BENCHMARK,
    "run_v3": RUN_FULL_V3_BENCHMARK,
})

## Strategy inventory

Wrappers that call the same numerical implementation are not treated as independent methods. The multiharmonic Fourier searches are represented by their Lomb--Scargle power; the redundant AoV transform is omitted.

In [ ]:
strategy_inventory = pd.DataFrame([
    ("catalog_consensus", "routing", "none", "Integrated Gaia EB / VSX / ASAS-SN variable / ZTF / OGLE consensus", "coverage only against catalog reference"),
    ("stored_lsp", "single method", "stored", "Astropy Lomb–Scargle peak from the July 1 backfill", "selected period"),
    ("stored_pdm", "single method", "stored", "MALCA Plavchan PDM", "selected period"),
    ("stored_ce", "single method", "stored", "MALCA conditional entropy", "selected period"),
    ("stored_long_ls", "single method", "stored", "Baseline-adaptive long-period Astropy LS", "selected period"),
    ("stored_event_period", "single method", "stored", "Detected-event spacing", "selected period where events exist"),
    ("production_consensus", "production ensemble", "fresh or stored", "Dip-focused PDM + CE + long LS + event/harmonic arbitration", "selected period plus abstention"),
    ("ls_core_pool", "candidate ensemble", "fresh", "Short/long LS", "candidate ceiling; selector evaluated separately"),
    ("classical_core_pool", "candidate ensemble", "fresh", "Short/long LS + classic PDM + CE", "candidate ceiling; selector evaluated separately"),
    ("v3_global_pool", "candidate ensemble", "fresh", "LS + Plavchan PDM + CE + BLS + multiharmonic + string/smoother/event proposers", "candidate ceiling"),
    ("v3_deterministic", "selector", "fresh", "V3 candidates and fixed-period features with documented deterministic ranking", "selected period"),
    ("v3_learned", "selector", "fresh", "V3 candidates and fixed-period features with saved LightGBM/scikit-learn ranker", "selected period and status"),
    ("hierarchical_v2", "selector", "not yet fitted at scale", "Harmonic-family ranker + P/2/P/2P resolver + independent acceptance model", "evaluation code exists; no completed expensive discovery fit"),
], columns=["strategy", "kind", "availability", "implementation", "reported_quantity"])
display(strategy_inventory)

## Real-source reference sets

The primary real-source reference requires a finite integrated catalog consensus, `period_consensus_agree`, and no conflict flag. A stricter multi-catalog slice requires at least two sources. These are external agreement benchmarks, not absolute truth. They are appropriate for testing internally computed periods, but not for scoring the catalog strategy itself.

In [ ]:
real_periods = add_catalog_reference(load_review_period_snapshot(REVIEW_DB))
real_periods["lc_exists"] = real_periods["lc_path"].map(lambda value: Path(str(value)).is_file() if pd.notna(value) else False)
real_accounting = pd.DataFrame([{
    "all_candidates": len(real_periods),
    "magnitude_12_15": int(real_periods["median_mag"].between(12.0, 15.0, inclusive="both").sum()),
    "light_curve_exists": int(real_periods["lc_exists"].sum()),
    "clean_catalog_reference": int(real_periods["catalog_reference_available"].sum()),
    "clean_multi_catalog_reference": int(real_periods["catalog_reference_tier"].eq("clean_multi_catalog").sum()),
    "stored_production_period": int(pd.to_numeric(real_periods["periodicity_period"], errors="coerce").notna().sum()),
}])
display(real_accounting)
if WRITE_AGGREGATES:
    real_accounting.to_parquet(OUTPUT_ROOT / "real_accounting.parquet", index=False)

In [ ]:
stored_summary = evaluate_stored_strategies(
    real_periods,
    tolerance=MATCH_TOLERANCE,
    harmonic_factors=DEFAULT_HARMONIC_FACTORS,
)
if WRITE_AGGREGATES:
    stored_summary.to_parquet(OUTPUT_ROOT / "stored_strategy_summary.parquet", index=False)
stored_display = stored_summary.sort_values(
    ["reference_independent", "family_agreement_conditional"],
    ascending=[False, False],
)
display(stored_display[[
    "strategy", "coverage_all", "coverage_on_reference",
    "exact_agreement_conditional", "family_agreement_conditional",
    "exact_yield_on_reference", "family_yield_on_reference",
]])

## Stratified fresh-computation sample

Runtime depends strongly on observation count. The sample balances catalog-reference availability, magnitude, and observation-count strata and stores inverse sampling weights. Set `MALCA_PERIOD_FRONTIER_SAMPLE_N=22942` only after the small run has completed cleanly. Six-machine execution can use `MALCA_PERIOD_FRONTIER_SHARD_COUNT=6` and a distinct `MALCA_PERIOD_FRONTIER_SHARD_INDEX` on each machine.

In [ ]:
runtime_sample = stratified_runtime_sample(
    real_periods.loc[real_periods["lc_exists"]],
    FRESH_SAMPLE_N,
    magnitude_min=12.0,
    magnitude_max=15.0,
)
runtime_sample = runtime_sample.iloc[
    np.arange(len(runtime_sample)) % SHARD_COUNT == SHARD_INDEX
].reset_index(drop=True)
runtime_sample.to_parquet(
    OUTPUT_ROOT / f"runtime_sample_shard_{SHARD_INDEX:03d}_of_{SHARD_COUNT:03d}.parquet",
    index=False,
)
display(runtime_sample[["candidate_id", "median_mag", "n_points", "catalog_reference_period", "sample_weight"]].head())
print("rows in this shard:", len(runtime_sample))

In [ ]:
v3_manifest = json.loads((V3_ROOT / "run_manifest.json").read_text())
V3_CONFIG = PeriodCandidateMethodsConfig(**v3_manifest["method_config"])
V3_RANKER = None
if RUN_FULL_V3_BENCHMARK:
    try:
        V3_RANKER = load_candidate_ranker_artifact(V3_ROOT / "models/candidate_ranker.joblib")
    except ValueError as exc:
        # The saved 2026-07-25 artifact predates the optional sample_weight_col=None
        # metadata field. Accept only that exact no-op schema addition; preserve every
        # other loader validation failure.
        if str(exc) != "Candidate-ranker configuration does not match metadata":
            raise
        import joblib
        from malca.evaluation.period_candidate_ranker import _metadata_config, _model_fingerprint
        candidate_ranker = joblib.load(V3_ROOT / "models/candidate_ranker.joblib")
        actual_config = _metadata_config(candidate_ranker.config)
        stored_config = dict(candidate_ranker.metadata.get("config", {}))
        no_op_fields = {
            key for key in set(actual_config).difference(stored_config)
            if actual_config[key] is None
        }
        normalized_actual = {key: value for key, value in actual_config.items() if key not in no_op_fields}
        if normalized_actual != stored_config:
            raise
        if _model_fingerprint(candidate_ranker.model) != candidate_ranker.metadata.get("model_fingerprint"):
            raise ValueError("Candidate-ranker fitted model fingerprint mismatch")
        calibrators = {
            "candidate": candidate_ranker.candidate_calibrator,
            "acceptance": candidate_ranker.acceptance_calibrator,
            "exact": candidate_ranker.exact_calibrator,
            "family": candidate_ranker.family_calibrator,
        }
        if {name: calibrator.metadata() for name, calibrator in calibrators.items()} != candidate_ranker.metadata.get("calibrators"):
            raise ValueError("Candidate-ranker calibrator state does not match metadata")
        V3_RANKER = candidate_ranker
        print(f"Loaded legacy V3 ranker after normalizing absent None fields: {sorted(no_op_fields)}")

COMPONENTS = {
    "ls_short": (("ls_short",), None),
    "ls_long": (("ls_long",), None),
    "pdm_classic": (("pdm",), "classic"),
    "pdm_plavchan": (("pdm",), "plavchan"),
    "conditional_entropy": (("ce",), None),
    "bls_coarse": (("bls_coarse",), None),
    "bls_adaptive": (("bls_adaptive",), None),
    "multiharmonic_ls_2": (("multiharmonic_ls_2",), None),
    "multiharmonic_ls_3": (("multiharmonic_ls_3",), None),
    "lafler_kinman": (("lafler_kinman",), None),
    "supersmoother": (("supersmoother",), None),
    "event_comb": (("event_comb",), None),
}


def load_prepared_light_curve(record: dict) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    path = Path(str(record["lc_path"]))
    df_g, df_v = read_lc_dat2(path.stem, str(path.parent), file_ext=path.suffix.lstrip("."))
    frame = pd.concat([df_g, df_v], ignore_index=True)
    frame = prepare_periodicity_lightcurve(frame)
    frame, _ = align_v_to_g_magnitude(frame)
    frame = frame.sort_values("JD", kind="stable")
    return (
        frame["JD"].to_numpy(dtype=float),
        frame["mag"].to_numpy(dtype=float),
        frame["error"].to_numpy(dtype=float),
    )


def event_epochs_from_record(record: dict) -> list[float]:
    return [epoch.center_jd for epoch in parse_run_epochs_json(record.get("dip_run_epochs_json"))]


def match_summary(periods: list[float], reference: object) -> tuple[bool, bool]:
    try:
        reference_value = float(reference)
    except (TypeError, ValueError):
        return False, False
    periods = [float(value) for value in periods if np.isfinite(value) and value > 0]
    if not periods or not np.isfinite(reference_value) or reference_value <= 0:
        return False, False
    matched = period_match_arrays(
        periods,
        np.full(len(periods), reference_value),
        tolerance=MATCH_TOLERANCE,
    )
    return bool(matched["is_exact"].any()), bool(matched["is_harmonic_family"].any())


def finite_positive(value: object) -> bool:
    try:
        number = float(value)
    except (TypeError, ValueError):
        return False
    return bool(np.isfinite(number) and number > 0)


def weighted_average(group: pd.DataFrame, column: str) -> float:
    values = pd.to_numeric(group[column], errors="coerce").to_numpy(dtype=float)
    weight_values = group["sample_weight"] if "sample_weight" in group else pd.Series(1.0, index=group.index)
    weights = pd.to_numeric(weight_values, errors="coerce").to_numpy(dtype=float)
    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    return float(np.average(values[valid], weights=weights[valid])) if valid.any() else np.nan


def load_shard_results(pattern: str, key_columns: list[str]) -> pd.DataFrame:
    files = sorted(OUTPUT_ROOT.glob(pattern))
    if not files:
        return pd.DataFrame()
    return (
        pd.concat([pd.read_parquet(path) for path in files], ignore_index=True)
        .drop_duplicates(key_columns, keep="last")
    )


def atomic_checkpoint(path: Path, rows: list[dict], key_columns: list[str]) -> None:
    incoming = pd.DataFrame(rows)
    if incoming.empty:
        return
    existing = pd.read_parquet(path) if path.is_file() else pd.DataFrame()
    combined = pd.concat([existing, incoming], ignore_index=True)
    combined = combined.drop_duplicates(key_columns, keep="last")
    temporary = path.with_name(f".{path.name}.tmp.parquet")
    combined.to_parquet(temporary, index=False)
    temporary.replace(path)


def run_in_batches(tasks: list[tuple], runner, checkpoint: Path, key_columns: list[str]) -> pd.DataFrame:
    existing = pd.read_parquet(checkpoint) if checkpoint.is_file() else pd.DataFrame()
    existing_keys = set(map(tuple, existing[key_columns].astype(str).to_numpy())) if not existing.empty else set()
    pending = [task for task in tasks if tuple(map(str, task[:len(key_columns)])) not in existing_keys]
    for start in range(0, len(pending), BATCH_SIZE):
        batch = pending[start:start + BATCH_SIZE]
        with parallel_config(
            backend="loky",
            n_jobs=WORKERS,
            inner_max_num_threads=INNER_THREADS_PER_WORKER,
        ):
            outputs = Parallel(batch_size=1, pre_dispatch="2*n_jobs")(
                delayed(runner)(*task) for task in batch
            )
        rows = [row for output in outputs for row in (output if isinstance(output, list) else [output])]
        atomic_checkpoint(checkpoint, rows, key_columns)
        print(f"checkpointed {min(start + len(batch), len(pending))}/{len(pending)} pending tasks")
    return pd.read_parquet(checkpoint) if checkpoint.is_file() else pd.DataFrame()

## Individual component cost and top-candidate accuracy

Each component is timed after canonical light-curve preparation. `top1_*` evaluates the first candidate from the named method; `oracle_*` asks whether any retained top-K proposal matches.

In [ ]:
def benchmark_component(candidate_id: str, component_name: str, record: dict) -> dict:
    methods, pdm_method = COMPONENTS[component_name]
    prepare_started = time.perf_counter()
    jd, mag, err = load_prepared_light_curve(record)
    preparation_seconds = time.perf_counter() - prepare_started
    config = replace(
        V3_CONFIG,
        enabled_global_methods=tuple(methods),
        pdm_method=pdm_method or V3_CONFIG.pdm_method,
    )
    search_started = time.perf_counter()
    searches = run_global_period_searches(
        jd,
        mag,
        err,
        event_epochs=event_epochs_from_record(record),
        config=config,
        methods=methods,
    )
    search_seconds = time.perf_counter() - search_started
    candidates = [candidate for result in searches.values() for candidate in result.candidates]
    top_period = float(candidates[0].period_days) if candidates else np.nan
    top_exact, top_family = match_summary([top_period], record.get("catalog_reference_period"))
    oracle_exact, oracle_family = match_summary(
        [candidate.period_days for candidate in candidates],
        record.get("catalog_reference_period"),
    )
    has_reference = finite_positive(record.get("catalog_reference_period"))
    return {
        "candidate_id": candidate_id,
        "component": component_name,
        "n_points_prepared": len(jd),
        "baseline_days": float(np.ptp(jd)),
        "preparation_seconds": preparation_seconds,
        "search_seconds": search_seconds,
        "selected_period_days": top_period,
        "candidate_count": len(candidates),
        "top1_exact": top_exact if has_reference else np.nan,
        "top1_family": top_family if has_reference else np.nan,
        "oracle_exact": oracle_exact if has_reference else np.nan,
        "oracle_family": oracle_family if has_reference else np.nan,
        "has_reference": has_reference,
        "sample_weight": float(record.get("sample_weight", 1.0)),
        "status": "|".join(f"{name}:{result.status}" for name, result in searches.items()),
    }


component_checkpoint = OUTPUT_ROOT / f"component_results_shard_{SHARD_INDEX:03d}_of_{SHARD_COUNT:03d}.parquet"
if RUN_COMPONENT_BENCHMARK:
    component_tasks = [
        (str(record["candidate_id"]), component_name, dict(record))
        for _, record in runtime_sample.iterrows()
        for component_name in COMPONENTS
    ]
    component_results = run_in_batches(
        component_tasks,
        benchmark_component,
        component_checkpoint,
        ["candidate_id", "component"],
    )
else:
    component_results = pd.read_parquet(component_checkpoint) if component_checkpoint.is_file() else pd.DataFrame()
    print("Component benchmark disabled; set MALCA_PERIOD_FRONTIER_RUN_COMPONENTS=1 to run.")

component_results = load_shard_results("component_results_shard_*.parquet", ["candidate_id", "component"])
if not component_results.empty:
    component_rows = []
    for component, group in component_results.groupby("component", sort=True):
        component_rows.append({
            "component": component,
            "n": len(group),
            "median_search_seconds": float(group["search_seconds"].median()),
            "median_preparation_seconds": float(group["preparation_seconds"].median()),
            "coverage": weighted_average(group.assign(_available=pd.to_numeric(group["selected_period_days"], errors="coerce").notna().astype(float)), "_available"),
            "top1_exact": weighted_average(group, "top1_exact"),
            "top1_family": weighted_average(group, "top1_family"),
            "oracle_exact": weighted_average(group, "oracle_exact"),
            "oracle_family": weighted_average(group, "oracle_family"),
        })
    component_summary = pd.DataFrame(component_rows).sort_values("median_search_seconds")
    display(component_summary)

    # These are candidate-generation ceilings, not operational selected-period accuracies.
    pool_components = {
        "ls_core_pool": ("ls_short", "ls_long"),
        "classical_core_pool": ("ls_short", "ls_long", "pdm_classic", "conditional_entropy"),
        "production_core_pool": ("pdm_plavchan", "conditional_entropy", "ls_long", "event_comb"),
        "v3_global_pool": (
            "ls_short", "ls_long", "pdm_plavchan", "conditional_entropy",
            "bls_coarse", "multiharmonic_ls_2", "multiharmonic_ls_3",
            "lafler_kinman", "supersmoother", "event_comb",
        ),
    }
    pool_rows = []
    for pool_name, members in pool_components.items():
        subset = component_results.loc[component_results["component"].isin(members)].copy()
        subset["_available"] = pd.to_numeric(subset["selected_period_days"], errors="coerce").notna().astype(float)
        per_source = subset.groupby("candidate_id", as_index=False).agg(
            n_components=("component", "nunique"),
            cpu_seconds_per_source=("search_seconds", "sum"),
            coverage=("_available", "max"),
            oracle_exact=("oracle_exact", "max"),
            oracle_family=("oracle_family", "max"),
            sample_weight=("sample_weight", "first"),
        )
        per_source = per_source.loc[per_source["n_components"].eq(len(members))]
        if per_source.empty:
            continue
        median_cpu_seconds = float(per_source["cpu_seconds_per_source"].median())
        pool_rows.append({
            "pool": pool_name,
            "components": ", ".join(members),
            "n": len(per_source),
            "median_cpu_seconds_per_source": median_cpu_seconds,
            "coverage": weighted_average(per_source, "coverage"),
            "candidate_oracle_exact": weighted_average(per_source, "oracle_exact"),
            "candidate_oracle_family": weighted_average(per_source, "oracle_family"),
            "projected_days_17m_at_80pct": project_runtime_days(
                n_sources=SURVEY_SOURCE_COUNT,
                seconds_per_source=median_cpu_seconds,
                workers_per_machine=PROJECTION_WORKERS_PER_MACHINE,
                machines=PROJECTION_MACHINES,
                parallel_efficiency=0.8,
            ),
        })
    pool_summary = pd.DataFrame(pool_rows)
    if not pool_summary.empty:
        pool_summary = pool_summary.sort_values("median_cpu_seconds_per_source")
        if WRITE_AGGREGATES:
            pool_summary.to_parquet(OUTPUT_ROOT / "candidate_pool_summary.parquet", index=False)
        display(pool_summary)


## Operational selectors: production consensus and full V3

These are the fair end-to-end comparisons because each returns one period or abstains. V3 reports both the documented deterministic baseline and the saved learned selector from the completed discovery run. The saved model was calibrated on simulations; its probabilities must not be interpreted as calibrated probabilities for the real ASAS-SN population without external calibration.

In [ ]:
def benchmark_production(candidate_id: str, selector: str, record: dict) -> dict:
    started = time.perf_counter()
    result, message = run_pipeline_period_search_for_payload(
        record,
        plot_dir=RUN_ROOT / "plots",
        min_period=0.1,
        max_period=5000.0,
        n_bootstrap=0,
    )
    elapsed = time.perf_counter() - started
    period = float(result.get("best_period", np.nan)) if result else np.nan
    exact, family = match_summary([period], record.get("catalog_reference_period"))
    has_reference = finite_positive(record.get("catalog_reference_period"))
    return {
        "candidate_id": candidate_id,
        "selector": selector,
        "runtime_seconds": elapsed,
        "selected_period_days": period,
        "solution_status": "selected" if np.isfinite(period) else "abstain",
        "exact_agreement": exact if has_reference else np.nan,
        "family_agreement": family if has_reference else np.nan,
        "has_reference": has_reference,
        "sample_weight": float(record.get("sample_weight", 1.0)),
        "message": message,
    }


def v3_candidate_score_frame(record: dict) -> tuple[pd.DataFrame, list[float]]:
    jd, mag, err = load_prepared_light_curve(record)
    event_epochs = event_epochs_from_record(record)
    baseline = float(np.ptp(jd))
    searches = run_global_period_searches(
        jd, mag, err,
        event_epochs=event_epochs,
        config=V3_CONFIG,
        methods=V3_CONFIG.enabled_global_methods,
    )
    expanded = build_candidate_bank(
        searches,
        baseline_days=baseline,
        config=V3_CONFIG,
        expand_harmonics=True,
    )
    shortlist = select_scoring_shortlist(expanded, config=V3_CONFIG)
    scores = score_candidate_bank(
        jd, mag, shortlist, err,
        event_epochs=event_epochs,
        config=V3_CONFIG,
    )
    refined = refine_scored_candidates(
        shortlist,
        scores,
        baseline_days=baseline,
        time=jd,
        mag=mag,
        err=err,
        config=V3_CONFIG,
        include_original=False,
    )
    refined_scores = score_candidate_bank(
        jd, mag, refined, err,
        event_epochs=event_epochs,
        config=V3_CONFIG,
    )
    rows = []
    for stage, stage_scores in (("expanded", scores), ("refined", refined_scores)):
        for index, score in enumerate(stage_scores):
            row = score.to_record()
            row.update({
                "candidate_id": f"{record['candidate_id']}::{stage}::{index:04d}",
                "base_view_id": str(record["candidate_id"]),
                "base_trial_id": str(record["candidate_id"]),
                "candidate_stage": stage,
                "baseline_days": baseline,
            })
            rows.append(row)
    candidate_periods = [candidate.period_days for candidate in expanded] + [candidate.period_days for candidate in refined]
    return pd.DataFrame(rows), candidate_periods


def benchmark_v3(candidate_id: str, selector: str, record: dict) -> list[dict]:
    started = time.perf_counter()
    frame, candidate_periods = v3_candidate_score_frame(record)
    candidate_seconds = time.perf_counter() - started
    has_reference = finite_positive(record.get("catalog_reference_period"))
    if frame.empty:
        return [{
            "candidate_id": candidate_id,
            "selector": name,
            "runtime_seconds": candidate_seconds,
            "selected_period_days": np.nan,
            "solution_status": "abstain",
            "exact_agreement": False if has_reference else np.nan,
            "family_agreement": False if has_reference else np.nan,
            "candidate_oracle_exact": False if has_reference else np.nan,
            "candidate_oracle_family": False if has_reference else np.nan,
            "has_reference": has_reference,
            "sample_weight": float(record.get("sample_weight", 1.0)),
        } for name in ("v3_deterministic", "v3_learned")]

    deterministic = rank_deterministic_baseline(frame, group_col="base_view_id")
    deterministic_row = deterministic.loc[deterministic["baseline_rank"].eq(1)].iloc[0]
    inference_started = time.perf_counter()
    learned_scores = score_candidate_ranker(frame, V3_RANKER)
    learned_solution = select_trial_solutions(learned_scores, V3_RANKER).iloc[0]
    inference_seconds = time.perf_counter() - inference_started
    oracle_exact, oracle_family = match_summary(candidate_periods, record.get("catalog_reference_period"))
    outputs = []
    for name, period, status, runtime in (
        ("v3_deterministic", float(deterministic_row["period_days"]), "selected", candidate_seconds),
        ("v3_learned", float(learned_solution["selected_period_days"]), str(learned_solution["solution_status"]), candidate_seconds + inference_seconds),
    ):
        exact, family = match_summary([period], record.get("catalog_reference_period"))
        outputs.append({
            "candidate_id": candidate_id,
            "selector": name,
            "runtime_seconds": runtime,
            "model_inference_seconds": inference_seconds if name == "v3_learned" else 0.0,
            "selected_period_days": period,
            "solution_status": status,
            "exact_agreement": exact if has_reference else np.nan,
            "family_agreement": family if has_reference else np.nan,
            "candidate_oracle_exact": oracle_exact if has_reference else np.nan,
            "candidate_oracle_family": oracle_family if has_reference else np.nan,
            "has_reference": has_reference,
            "sample_weight": float(record.get("sample_weight", 1.0)),
        })
    return outputs


selector_checkpoint = OUTPUT_ROOT / f"selector_results_shard_{SHARD_INDEX:03d}_of_{SHARD_COUNT:03d}.parquet"
selector_results = pd.read_parquet(selector_checkpoint) if selector_checkpoint.is_file() else pd.DataFrame()
if RUN_PRODUCTION_BENCHMARK:
    production_tasks = [(str(record["candidate_id"]), "production_consensus_0boot", dict(record)) for _, record in runtime_sample.iterrows()]
    selector_results = run_in_batches(
        production_tasks, benchmark_production, selector_checkpoint, ["candidate_id", "selector"]
    )
if RUN_FULL_V3_BENCHMARK:
    v3_tasks = [(str(record["candidate_id"]), "v3_learned", dict(record)) for _, record in runtime_sample.head(V3_SAMPLE_N).iterrows()]
    selector_results = run_in_batches(
        v3_tasks, benchmark_v3, selector_checkpoint, ["candidate_id", "selector"]
    )
if not RUN_PRODUCTION_BENCHMARK and not RUN_FULL_V3_BENCHMARK:
    print("Selector benchmarks disabled; enable the production and/or V3 environment flags to run.")

selector_results = load_shard_results("selector_results_shard_*.parquet", ["candidate_id", "selector"])
if not selector_results.empty:
    selector_rows = []
    for selector_name, group in selector_results.groupby("selector", sort=True):
        availability = pd.to_numeric(group["selected_period_days"], errors="coerce").notna().astype(float)
        selector_rows.append({
            "selector": selector_name,
            "n": len(group),
            "median_runtime_seconds": float(group["runtime_seconds"].median()),
            "coverage": weighted_average(group.assign(_available=availability), "_available"),
            "exact_agreement": weighted_average(group, "exact_agreement"),
            "family_agreement": weighted_average(group, "family_agreement"),
            "candidate_oracle_exact": weighted_average(group, "candidate_oracle_exact") if "candidate_oracle_exact" in group else np.nan,
            "candidate_oracle_family": weighted_average(group, "candidate_oracle_family") if "candidate_oracle_family" in group else np.nan,
        })
    selector_summary = pd.DataFrame(selector_rows).sort_values("median_runtime_seconds")
    display(selector_summary)

## Injection truth: non-circular recovery

This section uses the completed V3 simulation artifacts. It evaluates rank-1 raw proposer periods and the learned selected solution against known injected periods. This is the primary truth-based accuracy panel; it should be interpreted alongside the real-source catalog-agreement panel because simulation realism remains a separate assumption.

In [ ]:
raw_columns = ["base_trial_id", "base_view_id", "split", "event_mode", "method", "rank", "period_days"]
base_columns = ["base_trial_id", "base_view_id", "split", "event_mode", "is_signal", "true_period_days", "realized_baseline_days"]
raw_candidates = pd.read_parquet(V3_ROOT / "raw_method_candidates.parquet", columns=raw_columns)
injection_bases = pd.read_parquet(V3_ROOT / "base_results.parquet", columns=base_columns)
raw_top1 = (
    raw_candidates.sort_values(["base_view_id", "method", "rank"], kind="stable")
    .groupby(["base_view_id", "method"], as_index=False)
    .first()
    .merge(injection_bases, on=["base_trial_id", "base_view_id", "split", "event_mode"], how="left", validate="many_to_one")
)
raw_top1 = label_candidates(
    raw_top1,
    period_col="period_days",
    truth_col="true_period_days",
    tolerance=0.05,
    exact_tolerance=0.05,
    truth_periodic_col="is_signal",
    baseline_days="realized_baseline_days",
    rayleigh_tolerance=1.0,
)
saved_solutions = pd.read_parquet(V3_ROOT / "predictions/trial_solutions.parquet")
evaluation_roles = saved_solutions[["base_view_id", "evaluation_role"]].drop_duplicates()
raw_top1 = raw_top1.merge(evaluation_roles, on="base_view_id", how="left", validate="many_to_one")
injection_method_summary = (
    raw_top1.loc[
        raw_top1["split"].eq("validation")
        & raw_top1["event_mode"].eq("detected")
        & raw_top1["is_signal"].fillna(False)
    ]
    .groupby(["evaluation_role", "method"], as_index=False)
    .agg(n=("base_view_id", "size"), exact_recovery=("is_exact", "mean"), family_recovery=("is_harmonic_family", "mean"))
    .assign(accepted_fraction=np.nan, null_accepted_rate=np.nan)
    .sort_values(["evaluation_role", "exact_recovery"], ascending=[True, False])
)

# Use the official recovery table because recovery requires both accepting a
# solution and matching truth. The candidate's latent match while abstaining
# is not counted as recovered.
paper_recovery = pd.read_parquet(V3_ROOT / "metrics/paper_recovery_by_event_mode.parquet")
v3_solution_truth = paper_recovery.loc[
    paper_recovery["split"].eq("validation")
    & paper_recovery["event_mode"].eq("detected"),
    [
        "evaluation_role", "n_periodic", "exact_recovery_rate",
        "family_recovery_rate", "periodic_coverage", "null_accepted_rate",
        "exact_recovery_ci_low", "exact_recovery_ci_high",
        "family_recovery_ci_low", "family_recovery_ci_high",
    ],
].rename(columns={
    "n_periodic": "n",
    "exact_recovery_rate": "exact_recovery",
    "family_recovery_rate": "family_recovery",
    "periodic_coverage": "accepted_fraction",
})
v3_solution_truth.insert(1, "method", "v3_learned_selected")
injection_truth_summary = pd.concat([injection_method_summary, v3_solution_truth], ignore_index=True, sort=False)
display(injection_truth_summary.sort_values(["evaluation_role", "exact_recovery"], ascending=[True, False]))

## Frontier and survey-scale projection

The final plot is generated only from strategies with both measured runtime and non-circular selected-period agreement. Candidate-oracle accuracy is displayed separately and must never be substituted for the selected-period result. Survey projections explicitly show ideal, 80%, and 60% parallel efficiency for six machines with 60 workers each.

In [ ]:
frontier_parts = []
if "component_summary" in globals() and not component_results.empty:
    frontier_parts.append(component_summary.rename(columns={
        "component": "strategy",
        "median_search_seconds": "seconds_per_source",
        "top1_exact": "exact_accuracy",
        "top1_family": "family_accuracy",
    }).assign(result_kind="single_method_top1"))
if "selector_summary" in globals() and not selector_results.empty:
    frontier_parts.append(selector_summary.rename(columns={
        "selector": "strategy",
        "median_runtime_seconds": "seconds_per_source",
        "exact_agreement": "exact_accuracy",
        "family_agreement": "family_accuracy",
    }).assign(result_kind="operational_selector"))

if frontier_parts:
    frontier = pd.concat(frontier_parts, ignore_index=True, sort=False)
    frontier = mark_pareto_frontier(
        frontier,
        cost_col="seconds_per_source",
        accuracy_col="exact_accuracy",
        coverage_col="coverage",
        minimum_coverage=0.0,
    )
    projections = []
    for _, row in frontier.iterrows():
        for efficiency in PROJECTION_EFFICIENCIES:
            projections.append({
                "strategy": row["strategy"],
                "parallel_efficiency": efficiency,
                "projected_days_17m": project_runtime_days(
                    n_sources=SURVEY_SOURCE_COUNT,
                    seconds_per_source=float(row["seconds_per_source"]),
                    workers_per_machine=PROJECTION_WORKERS_PER_MACHINE,
                    machines=PROJECTION_MACHINES,
                    parallel_efficiency=efficiency,
                ),
            })
    projection_table = pd.DataFrame(projections)
    if WRITE_AGGREGATES:
        frontier.to_parquet(OUTPUT_ROOT / "cost_accuracy_frontier.parquet", index=False)
        projection_table.to_parquet(OUTPUT_ROOT / "survey_runtime_projections.parquet", index=False)
    display(frontier.sort_values("seconds_per_source"))
    display(projection_table.pivot(index="strategy", columns="parallel_efficiency", values="projected_days_17m"))

    fig, ax = plt.subplots(figsize=(10, 7))
    for _, row in frontier.iterrows():
        ax.scatter(row["seconds_per_source"], row["exact_accuracy"], s=90 if row["is_pareto"] else 45, alpha=1.0 if row["is_pareto"] else 0.55)
        ax.annotate(str(row["strategy"]), (row["seconds_per_source"], row["exact_accuracy"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
    ax.set_xscale("log")
    ax.set_xlabel("Measured seconds per light curve (single worker)")
    ax.set_ylabel("Exact selected-period agreement")
    ax.set_title("Period-finding cost–accuracy frontier")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    if WRITE_AGGREGATES:
        fig.savefig(OUTPUT_ROOT / "cost_accuracy_frontier.png", dpi=180)
    plt.show()
else:
    print("No fresh timing results yet. Run at least one component or selector benchmark, then rerun this cell.")

## Decision rule

Read the tables in plain English as follows: `coverage` is how often a method returns anything; `exact` is how often that one returned answer is the injected/catalog period; `family` also gives credit for simple half/double/triple-period aliases; and `candidate oracle` asks only whether the right answer was somewhere in the method pool. Candidate oracle is an upper bound, not what the pipeline actually recovered. In the cost–accuracy plot, farther left is cheaper and higher is better.

Do not choose the numerically highest-accuracy row automatically. Prefer the cheapest Pareto strategy that satisfies predeclared requirements for:

- exact selected-period recovery;
- harmonic-family recovery;
- abstention/coverage;
- performance in long-period, narrow-dip, low-S/N, and alias-adjacent slices;
- and acceptable projected survey runtime.

A likely production architecture is catalog-first routing followed by a cheap internal ensemble, with the full V3 selector reserved for unresolved or scientifically valuable cases. This notebook is designed to test that hypothesis rather than assume it.